# SocialEngineerArena GRPO Training

Use this notebook on Hugging Face/Colab compute after credits are available. It is wired around the live OpenEnv reward function, not a static label-only dataset.

In [1]:
# If running from notebooks/, move to repo root first.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

!pip install -q -U pip
!pip install -q openenv-core trl transformers accelerate datasets
!pip install -q -e .

D:\Openenv-final\social-engineer-arena


In [2]:
import json
import os
import requests

from scripts.train_grpo_placeholder import format_prompt
from social_engineer_arena.models import ArenaAction
from social_engineer_arena.server.environment import SocialEngineerArenaEnvironment

env = SocialEngineerArenaEnvironment()

# Always require user-provided endpoint/token (no hardcoded private URL).
ENDPOINT_URL = os.getenv("HF_ENDPOINT_URL", "")
HF_TOKEN = os.getenv("HF_TOKEN", "")

if not ENDPOINT_URL:
    raise RuntimeError("Set HF_ENDPOINT_URL before running this notebook.")
if not HF_TOKEN:
    print("Set HF_TOKEN in your shell before running this cell.")
    print("PowerShell: $env:HF_TOKEN='hf_xxx'")

sample_obs = env.reset()
sample_prompt = format_prompt(sample_obs)
print("Endpoint:", ENDPOINT_URL)
print("Role:", sample_obs.role, "Scenario:", sample_obs.scenario_id)
print(sample_prompt[:700])

Endpoint: https://kq46ry5f293qor2g.us-east-1.aws.endpoints.huggingface.cloud
Role: defender Scenario: def-001-payroll-switch
You are playing SocialEngineerArena.
Role: defender
Persona: Payroll analyst at a mid-size robotics company
Organization: Northstar Robotics
Context: The CFO is traveling. The employee directory shows the real CFO uses carla.chen@northstar.example.
Message: From: Carla Chen <carla.chen-payroll@northstar.example>
Subject: urgent payroll correction before board call

I am about to board and cannot get into the HR portal. Update my direct deposit to the attached account today and do not loop anyone in until I am back. We will clean up paperwork later.
Task: Classify the incoming message as safe, phishing, or pretexting. Explain concrete evidence and list the subtle cues you used.

Return JSON w


### Endpoint rollout sanity check
Run the next **code cell** to generate actions, score in env, and track rewards.

In [ ]:
# Endpoint rollout sanity check: generate actions, score in env, track rewards.
def generate_action_json(prompt: str, max_new_tokens: int = 220) -> str:
    if not HF_TOKEN:
        raise RuntimeError("HF_TOKEN missing. Set $env:HF_TOKEN before running.")

    headers = {
        "Authorization": f"Bearer {HF_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "messages": [
            {"role": "system", "content": "Return only one JSON object matching the requested schema."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.7,
        "top_p": 0.9,
        "max_tokens": max_new_tokens,
    }

    response = requests.post(
        f"{ENDPOINT_URL}/v1/chat/completions",
        headers=headers,
        json=payload,
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    return data["choices"][0]["message"]["content"].strip()


def parse_action(completion: str) -> ArenaAction:
    try:
        return ArenaAction.model_validate_json(completion)
    except Exception:
        start = completion.find("{")
        end = completion.rfind("}")
        if start != -1 and end != -1 and end > start:
            return ArenaAction.model_validate_json(completion[start:end + 1])
        return ArenaAction(explanation=completion)


rewards = []
rows = []
for i in range(8):
    obs = env.reset()
    prompt = format_prompt(obs)
    completion = generate_action_json(prompt)
    action = parse_action(completion)
    _, reward, done = env.step(action)
    rewards.append(reward)
    rows.append(
        {
            "i": i,
            "role": obs.role,
            "scenario_id": obs.scenario_id,
            "reward": reward,
            "done": done,
            "verdict": action.verdict,
        }
    )

rows, sum(rewards) / len(rewards)

In [3]:
# Optional: lightweight training placeholder loop.
# This notebook now verifies end-to-end env scoring using your HF Inference Endpoint.
# For full RL (GRPO), keep the same reward path: env.reset() -> model completion -> ArenaAction -> env.step().

print("Endpoint verification complete if rewards are non-trivial and vary across scenarios.")
print("Next step: plug this reward pipeline into GRPOTrainer on GPU compute.")

Endpoint verification complete if rewards are non-trivial and vary across scenarios.
Next step: plug this reward pipeline into GRPOTrainer on GPU compute.
